## Sephora: products and reviews

### Step 1: look at the files
- Look at the Sephora files before cleaning: which files and columns exist, how big they are, and how the ingredient lists and skin types are written. The review text is not needed yet, so I load only the columns I need, which saves memory (the five review files have about 1.1 million rows).

#### Cell 1: the columns of every file (reads only the header, so it is fast)

In [1]:
import glob                                              # find files by pattern
import os                                                # file names
import pandas as pd                                      # tables

pd.set_option("display.max_columns", None)               # show all columns
pd.set_option("display.max_colwidth", 80)                # long texts are cut after 80 characters

folder = "../data/raw/sephora/"
for f in sorted(glob.glob(folder + "*.csv")):            # every CSV in the Sephora folder
    cols = pd.read_csv(f, nrows=0).columns.tolist()      # nrows=0 reads only the column names
    print(os.path.basename(f), "->", len(cols), "columns")
    print("   ", cols, "\n")

product_info.csv -> 27 columns
    ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price'] 

reviews_0-250.csv -> 19 columns
    ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd'] 

reviews_1250-end.csv -> 19 columns
    ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_cou

#### Cell 2: load products and reviews (selected columns) and look at them

In [2]:
# 1. Products: whole file
products = pd.read_csv(folder + "product_info.csv")
print("products:", products.shape)
print(products[["product_id", "product_name", "brand_name", "primary_category", "rating", "reviews", "ingredients"]]
      .sample(3, random_state=1))

# 2. Reviews: five files, only the columns we need now (no review text yet)
need = ["author_id", "rating", "is_recommended", "submission_time", "skin_type", "skin_tone", "product_id"]
review_files = sorted(glob.glob(folder + "reviews_*.csv"))
reviews = pd.concat([pd.read_csv(f, usecols=need) for f in review_files], ignore_index=True)
print("\nreviews:", reviews.shape)

# 3. Missing values: how complete are the columns?
print("\nMissing values in products (top 6):")
print(products.isna().sum().sort_values(ascending=False).head(6))
print("\nMissing values in reviews:")
print(reviews.isna().sum())

# 4. The key columns for chapter 4
print("\nSkin types in reviews:")
print(reviews["skin_type"].value_counts(dropna=False))
print("\nRatings:")
print(reviews["rating"].value_counts().sort_index())
print("\nProduct categories:")
print(products["primary_category"].value_counts())

# 5. One ingredient list as written in the data
print("\nExample ingredient text:")
print(products["ingredients"].dropna().iloc[0][:500])

products: (8494, 27)
     product_id                                           product_name  \
3293    P375375                                     Angled Liner Brush   
3991    P434429  Resistance Strengthening Conditioner for Damaged Hair   
6965    P502169          Barrier+ Besties Hydrating and Nourishing Kit   

     brand_name primary_category  rating  reviews  \
3293  Hourglass           Makeup  4.5333     15.0   
3991  Kérastase             Hair  4.1678    143.0   
6965    Skinfix         Skincare  4.7162     74.0   

                                                                          ingredients  
3293                                                                              NaN  
3991  ['Water, Sodium Laureth Sulfate, Disodium Cocoamphodiacetate, Glycol Distear...  
6965  ['barrier+ Triple Lipid-Peptide Cream:', 'Water/Eau/Aqua, Caprylic/Capric Tr...  


C:\Users\simin\AppData\Local\Temp\ipykernel_50616\186489175.py:10: DtypeWarning: Columns (1: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.concat([pd.read_csv(f, usecols=need) for f in review_files], ignore_index=True)



reviews: (1094411, 7)

Missing values in products (top 6):
sale_price_usd     8224
value_price_usd    8043
variation_desc     7244
child_max_price    5740
child_min_price    5740
highlights         2207
dtype: int64

Missing values in reviews:
author_id               0
rating                  0
is_recommended     167988
submission_time         0
skin_tone          170539
skin_type          111557
product_id              0
dtype: int64

Skin types in reviews:
skin_type
combination    544513
dry            185937
normal         131910
oily           120494
NaN            111557
Name: count, dtype: int64

Ratings:
rating
1     61223
2     53032
3     81816
4    199389
5    698951
Name: count, dtype: int64

Product categories:
primary_category
Skincare           2420
Makeup             2369
Hair               1464
Fragrance          1432
Bath & Body         405
Mini Size           288
Men                  60
Tools & Brushes      52
Gifts                 4
Name: count, dtype: int64

Exampl

**Findings: Sephora files**
- 8,494 products (2,420 skincare), 1,094,411 reviews, 5 review files with the same 19 columns.
- Skin type is given by the reviewer: combination 50%, dry 17%, normal 12%, oily 11%, missing 10%.
- Ratings are strongly skewed to the top (64% five stars), so rank-based tests are appropriate.
- Ingredient lists are stored as text that looks like a Python list, with sub-lists and header lines for sets, and names differ from INCI spelling. They need parsing and normalising before matching with CosIng.
  

## Step 2: parse the ingredient lists (skincare only)

 - Turn the ingredient text into a table with one row per product and ingredient. The cell is a list written as text, so I convert it back to a list. Header lines of sets (ending with “:”) are skipped. I split at “, “ (comma plus space), not at the comma alone, because chemical names like “1,2-Hexanediol” contain commas. The position is kept, because ingredients are listed from the highest to the lowest amount.

- Fix the split. A comma inside parentheses, like “Water (Aqua, Eau)”, belongs to one ingredient. I walk through the text character by character, count open parentheses, and split at “, “ only when no parenthesis is open.

In [3]:
import ast, re, warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)   # a few texts contain a backslash, harmless

def split_outside_brackets(s):
    """Split text at ', ' but not when the comma is inside parentheses."""
    items, buf, depth = [], [], 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1                                       # entering a bracket
        elif ch == ")":
            depth = max(depth - 1, 0)                        # leaving a bracket (never below 0)
        if ch == "," and depth == 0 and s[i+1:i+2] == " ":   # comma + space outside brackets = new ingredient
            items.append("".join(buf)); buf = []
        else:
            buf.append(ch)
    items.append("".join(buf))
    return items

def parse_ingredients(text):
    """Ingredient text of one product -> list of single ingredient names."""
    if pd.isna(text):
        return []                                            # product without an ingredient list
    try:
        parts = ast.literal_eval(text)                       # the cell holds a Python list written as text
    except (ValueError, SyntaxError):
        parts = [text]                                       # fallback: treat the whole text as one list
    items = []
    for part in parts:
        part = part.strip()
        if part.endswith(":"):                               # header of a set component, skip
            continue
        items += [i.strip().rstrip(".") for i in split_outside_brackets(part)]
    return [i for i in items if i]
    
# Skincare products only (this line was missing in the replacement cell)
skin = products[products["primary_category"] == "Skincare"][
    ["product_id", "product_name", "brand_name", "rating", "reviews", "price_usd", "ingredients"]].copy()
skin["ingredient_list"] = skin["ingredients"].apply(parse_ingredients)
long = skin[["product_id", "ingredient_list"]].explode("ingredient_list").dropna()
long = long.rename(columns={"ingredient_list": "ingredient_raw"})
long["position"] = long.groupby("product_id").cumcount() + 1

print("without ingredient list:", (skin["ingredient_list"].str.len() == 0).sum())
print("rows:", len(long), "| distinct spellings:", long["ingredient_raw"].nunique())
print("median per product:", long.groupby("product_id").size().median())

# Check: very long "ingredients" mean a bracket was never closed and pieces got merged
too_long = long[long["ingredient_raw"].str.len() > 80]
print("items longer than 80 characters:", len(too_long))
print(too_long["ingredient_raw"].head(3).tolist())

# Check the earlier problem product
long[long["product_id"] == "P439055"].head(5)

without ingredient list: 134
rows: 87660 | distinct spellings: 8001
median per product: 33.0
items longer than 80 characters: 153
['Citrus Medica Limonum Fruit Extract (Citrus Medica Limonum (Lemon) Fruit Extract)', 'Citrus Aurantium Dulcis Fruit Extract (Citrus Aurantium Dulcis (Orange) Fruit Extract)', 'Phenoxyethanol. May Contain/Peut Contenir (+/-): Iron Oxides (Ci 77491, Ci 77492, Ci 77499)']


,product_id,ingredient_raw,position
89,P439055,Collagen (Vegan)*,1
89,P439055,"Water (Aqua, Eau)",2
89,P439055,Ethylhexyl Palmitate,3
89,P439055,Oryza Sativa (Rice) Bran Extract,4
89,P439055,Caprylic/Capric Triglyceride,5


## Step 3: normalise and match against CosIng

- Make Sephora spellings comparable with CosIng. I convert to UPPER case and remove marks like “”. If a name has a “May Contain” part, I cut it off. Then I try several candidates in order: the full name first (so “Caprylic/Capric Triglyceride” stays whole), then the part before the bracket, the text inside the bracket, and the pieces split at “/”. Water is called AQUA in CosIng, so I add this alias. The result is the match rate for the stop rule (85% of rows).*

In [4]:
import os, re
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. CosIng names from MySQL
load_dotenv("../.env", override=True)
engine = create_engine(os.getenv("CON_STRING"))
cosing = pd.read_sql("SELECT ingredient_id, inci_name FROM ingredients", engine)
cosing["key"] = cosing["inci_name"].str.upper().str.strip()
key2id = dict(zip(cosing["key"], cosing["ingredient_id"]))     # name -> id (one entry per name)

ALIASES = {"WATER": "AQUA", "EAU": "AQUA"}                      # CosIng uses AQUA for water

def clean(s):
    s = s.upper()
    s = re.sub(r"[*†‡°•]", "", s)                               # marks like organic stars
    s = re.sub(r"\s+", " ", s)                                  # collapse double spaces
    return s.strip(" .:;,")

def candidates(raw):
    """All spellings of one ingredient to try, best guess first."""
    s = clean(raw)
    s = re.split(r"\bMAY CONTAIN\b|PEUT CONTENIR|\+/-", s)[0].strip(" .:;,")   # cut off the "may contain" part
    cands = [s]                                                 # 1. full name first
    m = re.match(r"^(.*?)\s*\((.*)\)$", s)                      # "NAME (OTHER, NAME)"
    if m:
        cands += [m.group(1)] + [x.strip() for x in re.split(r"[,/]", m.group(2))]
    if "/" in s:
        cands += [x.strip() for x in s.split("/")]              # "WATER/EAU/AQUA"
    cands += [ALIASES[c] for c in cands if c in ALIASES]        # WATER -> AQUA
    return [c for c in cands if c]

def match(raw):
    for c in candidates(raw):
        if c in key2id:
            return key2id[c]                                    # first candidate found in CosIng
    return None

# 2. Match every distinct spelling once (8,000 instead of 87,660 rows)
spell = long["ingredient_raw"].value_counts().rename_axis("ingredient_raw").reset_index(name="n_rows")
spell["ingredient_id"] = spell["ingredient_raw"].apply(match)
long = long.merge(spell[["ingredient_raw", "ingredient_id"]], on="ingredient_raw", how="left")

# 3. Match rate
print("distinct spellings matched: %.1f%%" % (spell["ingredient_id"].notna().mean() * 100))
print("ROWS matched:               %.1f%%" % (long["ingredient_id"].notna().mean() * 100))
print("products with >=80%% of ingredients matched: %.1f%%" % (
    long.groupby("product_id")["ingredient_id"].apply(lambda s: s.notna().mean() >= 0.8).mean() * 100))

# 4. What is missing? Most frequent unmatched spellings
spell[spell["ingredient_id"].isna()].head(30)

distinct spellings matched: 59.1%
ROWS matched:               85.3%
products with >=80% of ingredients matched: 72.4%


,ingredient_raw,n_rows,ingredient_id
26,Helianthus Annuus (Sunflower) Seed Oil,384,NaN
45,Butyrospermum Parkii (Shea) Butter,274,NaN
71,Simmondsia Chinensis (Jojoba) Seed Oil,184,NaN
77,Rosmarinus Officinalis (Rosemary) Leaf Extract,167,NaN
79,Alcohol Denat,164,NaN
81,Fragrance,158,NaN
85,Cucumis Sativus (Cucumber) Fruit Extract,149,NaN
88,Glycyrrhiza Glabra (Licorice) Root Extract,145,NaN
93,Pyrus Malus (Apple) Fruit Extract,141,NaN
110,Curcuma Longa (Turmeric) Root Extract,126,NaN


### Step 3b: 
- Improve the matching. CosIng names have no common name in brackets, so I also try the name without the bracket. I clean the CosIng names the same way as the Sephora names (no trailing dot), add the alias FRAGRANCE -> PARFUM, and split at “” as well as “/”.

In [5]:
# 1. Clean the CosIng names the same way as the Sephora names (UPPER, no trailing dot)
cosing["key"] = cosing["inci_name"].apply(clean)
key2id = dict(zip(cosing["key"], cosing["ingredient_id"]))

ALIASES = {"WATER": "AQUA", "EAU": "AQUA", "FRAGRANCE": "PARFUM"}

def candidates(raw):
    """All spellings of one ingredient to try, best guess first."""
    s = clean(raw)
    s = re.split(r"\bMAY CONTAIN\b|PEUT CONTENIR|\+/-", s)[0].strip(" .:;,")   # cut off "may contain"
    cands = [s]                                                      # 1. full name
    no_brackets = re.sub(r"\s*\([^)]*\)", "", s)                     # 2. "HELIANTHUS ANNUUS (SUNFLOWER) SEED OIL" -> "HELIANTHUS ANNUUS SEED OIL"
    cands.append(re.sub(r"\s+", " ", no_brackets).strip())
    m = re.match(r"^(.*?)\s*\((.*)\)$", s)                           # 3. "NAME (OTHER, NAME)"
    if m:
        cands += [m.group(1)] + [x.strip() for x in re.split(r"[,/\\]", m.group(2))]
    if "/" in s or "\\" in s:
        cands += [x.strip() for x in re.split(r"[/\\]", s)]          # 4. "WATER/EAU/AQUA" and "WATER\AQUA\EAU"
    cands += [ALIASES[c] for c in cands if c in ALIASES]             # 5. WATER -> AQUA, FRAGRANCE -> PARFUM
    return [c for c in cands if c]

# 2. Match again (drop the old result first so the merge does not make double columns)
long = long.drop(columns="ingredient_id", errors="ignore")
spell["ingredient_id"] = spell["ingredient_raw"].apply(match)
long = long.merge(spell[["ingredient_raw", "ingredient_id"]], on="ingredient_raw", how="left")

# 3. Match rate
print("distinct spellings matched: %.1f%%" % (spell["ingredient_id"].notna().mean() * 100))
print("ROWS matched:               %.1f%%" % (long["ingredient_id"].notna().mean() * 100))
print("products with >=80%% of ingredients matched: %.1f%%" % (
    long.groupby("product_id")["ingredient_id"].apply(lambda s: s.notna().mean() >= 0.8).mean() * 100))

# 4. Most frequent remaining unmatched spellings
spell[spell["ingredient_id"].isna()].head(25)

distinct spellings matched: 72.8%
ROWS matched:               96.0%
products with >=80% of ingredients matched: 92.8%


,ingredient_raw,n_rows,ingredient_id
392,Octisalate 5%,38,NaN
417,Butyrospermum Parkii (Shea Butter),35,NaN
464,Avobenzone 3%,32,NaN
504,Hordeum Vulgare (Barley) Extract\Extrait D’orge,29,NaN
560,Hordeum Vulgare (Barley) Extract/Extrait D’orge,26,NaN
576,Iron Oxides,25,NaN
637,Olea Europaea (Olive) Oil*,23,NaN
701,Purified Water,20,NaN
859,Flavor,16,NaN
871,Octocrylene 10%,16,NaN


**Findings: Sephora ingredients**
- 2,420 skincare products; 134 (5.5%) have no ingredient list.
- 87,660 product-ingredient rows, median 33 ingredients per product.
- After normalising names (upper case, no brackets with common names, aliases like Water -> AQUA, Fragrance -> PARFUM), **96.0% of ingredient mentions match a CosIng INCI name**; 92.8% of products have at least 80% of ingredients matched.
- Unmatched names are mostly rare spellings (percentages, typos, French text). They stay unmatched; this is a known limitation.

## Step 4: clean the reviews and save the Sephora files

- Keep only reviews of skincare products and save three clean tables: products, ingredients (with the CosIng id) and reviews. Reviews without a skin type stay in the file, but the skin-type tests will use only reviews that have one. The date column is converted to a real date.

In [6]:
# 1. Reviews of skincare products only
skin_ids = set(skin["product_id"])                                   # the 2,420 skincare product ids
rev = reviews[reviews["product_id"].isin(skin_ids)].copy()
rev["submission_time"] = pd.to_datetime(rev["submission_time"])      # text -> date
rev["author_id"] = rev["author_id"].astype(str)                      # mixed types -> text (silences the warning)

# 2. Checks
print("skincare reviews:", len(rev), "| with skin type:", rev["skin_type"].notna().sum())
print("skincare products with at least one review:", rev["product_id"].nunique(), "of", len(skin))
print(rev["skin_type"].value_counts(dropna=False))
print("dates:", rev["submission_time"].min().date(), "to", rev["submission_time"].max().date())

# 3. Save three clean tables (data/clean is in .gitignore)
skin.drop(columns=["ingredients", "ingredient_list"]).to_csv("../data/clean/sephora_products.csv", index=False)

ing = long[["product_id", "position", "ingredient_raw", "ingredient_id"]].copy()
ing["ingredient_id"] = ing["ingredient_id"].astype("Int64")          # whole numbers, empty where no match
ing.to_csv("../data/clean/sephora_ingredients.csv", index=False)

rev[["author_id", "product_id", "rating", "is_recommended", "skin_type", "skin_tone",
     "submission_time"]].to_csv("../data/clean/sephora_reviews.csv", index=False)
print("saved:", len(skin), "products |", len(ing), "ingredient rows |", len(rev), "reviews")

skincare reviews: 1094411 | with skin type: 982854
skincare products with at least one review: 2351 of 2420
skin_type
combination    544513
dry            185937
normal         131910
oily           120494
NaN            111557
Name: count, dtype: int64
dates: 2008-08-28 to 2023-03-21
saved: 2420 products | 87660 ingredient rows | 1094411 reviews


**Findings: Sephora clean files**
- All 1,094,411 reviews belong to skincare products (the review files contain only skincare), covering 2,351 of 2,420 skincare products.
- 69 skincare products (2.9%) have no reviews (empty review count in the product table, none in the review files). They are excluded from the rating tests.
- 982,854 reviews (90%) have a skin type (combination, dry, normal, oily) and are the base for the skin-type tests.
- Reviews cover 2008 to March 2023, the California reports end in June 2020; the two sources are linked by ingredient, not by date.
- Saved: sephora_products.csv (2,420), sephora_ingredients.csv (87,660 rows with CosIng id), sephora_reviews.csv (1,094,411).

In [7]:
# Skincare products that have no review in the review files
no_rev = skin[~skin["product_id"].isin(rev["product_id"])]
print(len(no_rev))
print(no_rev["reviews"].describe())          # reviews count according to the product table
print((no_rev["reviews"].fillna(0) == 0).sum(), "of them have 0 or missing review count in the product table")

69
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: reviews, dtype: float64
69 of them have 0 or missing review count in the product table
